# 12 — Chow-Liu Bayesian Network Structure Learning

Learn joint dependency structure between columns using the Chow-Liu algorithm (max spanning tree of mutual information). Understand which columns are most dependent.

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.inference.tier3_research import ChowLiuNetwork
import importlib
from sqllocks_spindle.cli import _get_domain_registry

spindle = Spindle()
reg = _get_domain_registry()
mod, cls, _ = reg['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')
result = spindle.generate(domain=domain, scale='small', seed=42)
table = next(iter(result.tables))
df = result.tables[table]
print(f'Learning structure for: {table}  ({len(df)} rows, {len(df.columns)} cols)')

In [ ]:
net = ChowLiuNetwork(n_bins=10)
cl_result = net.fit(df)

print('\n--- Chow-Liu Tree Edges (by mutual information) ---')
for edge in sorted(cl_result.edges, key=lambda e: -e.mutual_information):
    print(f'  {edge.parent}  →  {edge.child}   MI={edge.mutual_information:.4f}')

In [ ]:
# Mutual information heatmap (text-based)
cols = cl_result.column_order
print('\n--- Mutual Information Matrix (top correlations) ---')
mi_pairs = []
for ci in cols:
    for cj, mi in cl_result.mutual_info_matrix[ci].items():
        mi_pairs.append((mi, ci, cj))
mi_pairs.sort(reverse=True)
print(f'{"Column A":<25} {"Column B":<25} {"MI":>10}')
print('-' * 62)
for mi, ci, cj in mi_pairs[:10]:
    print(f'{ci:<25} {cj:<25} {mi:>10.4f}')